# VisProbe: Adversarial Training Effectiveness Experiment

**GPU Runtime Required:** Runtime > Change runtime type > T4 GPU

This notebook tests:
1. Natural perturbations (Gaussian Blur)
2. Adversarial attacks (PGD)
3. Compositional attacks (Natural + Adversarial combined)

Expected time: ~30-45 minutes on T4 GPU

In [ ]:
# Cell 1: Install dependencies
!pip install -q torch torchvision
!pip install -q robustbench
!pip install -q git+https://github.com/fra31/auto-attack
!pip install -q adversarial-robustness-toolbox

# Install VisProbe from GitHub
!pip install -q git+https://github.com/bilgedemirkaya/VisProbe

print("All dependencies installed!")

In [ ]:
# Cell 2: Mount Google Drive (for ImageNet data)
from google.colab import drive
drive.mount('/content/drive')

# SET YOUR IMAGENET PATH HERE
# Example: '/content/drive/MyDrive/imagenet_val' or '/content/drive/MyDrive/datasets/imagenet/val'
IMAGENET_PATH = '/content/drive/MyDrive/imagenet_val'  # <-- UPDATE THIS PATH

import os
if os.path.exists(IMAGENET_PATH):
    num_classes = len([d for d in os.listdir(IMAGENET_PATH) if os.path.isdir(os.path.join(IMAGENET_PATH, d))])
    print(f"ImageNet found at: {IMAGENET_PATH}")
    print(f"Number of classes: {num_classes}")
else:
    print(f"ERROR: Path not found: {IMAGENET_PATH}")
    print("Please update IMAGENET_PATH to your ImageNet validation folder location")

In [ ]:
# Cell 3: Setup
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
import json
from datetime import datetime

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

NUM_SAMPLES = 500  # Number of samples to test

# ImageNet transforms
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

imagenet_val = ImageFolder(root=IMAGENET_PATH, transform=transform)
print(f"Loaded {len(imagenet_val)} images from {len(imagenet_val.classes)} classes")

In [ ]:
# Cell 4: Load models
from robustbench.utils import load_model

print("Loading vanilla ResNet50...")
vanilla = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
vanilla = vanilla.to(device).eval()

print("Loading adversarially-trained model (Salman2020Do_R50)...")
robust = load_model(
    model_name='Salman2020Do_R50',
    dataset='imagenet',
    threat_model='Linf'
)
robust = robust.to(device).eval()

print("Models loaded!")

In [ ]:
# Cell 5: Get mutually correct samples
def get_mutually_correct_samples(model1, model2, dataset, n=500):
    """Get samples correctly classified by BOTH models."""
    model1.eval()
    model2.eval()
    
    samples = []
    
    with torch.no_grad():
        for i in range(len(dataset)):
            if len(samples) >= n:
                break
            
            img, label = dataset[i]
            img_batch = img.unsqueeze(0).to(device)
            
            pred1 = model1(img_batch).argmax().item()
            pred2 = model2(img_batch).argmax().item()
            
            if pred1 == label and pred2 == label:
                samples.append((img, label))
            
            if i % 500 == 0:
                print(f"  Scanned {i} images, found {len(samples)} mutual correct...")
    
    return samples

print(f"Finding {NUM_SAMPLES} samples correctly classified by both models...")
samples = get_mutually_correct_samples(vanilla, robust, imagenet_val, n=NUM_SAMPLES)
print(f"\nFound {len(samples)} mutually correct samples")

In [ ]:
# Cell 6: Run ALL tests
from visprobe import search
from visprobe.strategies import (
    GaussianBlurStrategy,
    BrightnessStrategy,
    GaussianNoiseStrategy,
    PGDStrategy,
    Compose,
)

results = {"vanilla": {}, "robust": {}}

# ============================================================
# TEST 1: NATURAL PERTURBATIONS
# ============================================================
print("=" * 60)
print("TEST 1: NATURAL PERTURBATIONS (Gaussian Blur)")
print("=" * 60)

for model_name, model in [("vanilla", vanilla), ("robust", robust)]:
    print(f"\nTesting {model_name}...")
    report = search(
        model=model,
        data=samples,
        perturbation="gaussian_blur",
        level_lo=0.0,
        level_hi=10.0,
        device=device,
        normalization="imagenet"
    )
    results[model_name]["natural"] = report
    print(f"  {model_name}: threshold={report.search.get('threshold', 'N/A')}")

In [ ]:
# Cell 7: TEST 2 - ADVERSARIAL ATTACKS
print("\n" + "=" * 60)
print("TEST 2: ADVERSARIAL ATTACKS (PGD)")
print("=" * 60)

for model_name, model in [("vanilla", vanilla), ("robust", robust)]:
    print(f"\nTesting {model_name}...")
    report = search(
        model=model,
        data=samples,
        strategy=lambda eps: PGDStrategy(eps=eps, eps_step=max(eps/10, 1e-6), max_iter=20),
        level_lo=0.0,
        level_hi=0.03,
        device=device,
        normalization="imagenet",
        strategy_name="PGD Attack"
    )
    results[model_name]["adversarial"] = report
    print(f"  {model_name}: threshold={report.search.get('threshold', 'N/A')}")

In [ ]:
# Cell 8: TEST 3 - COMPOSITIONAL ATTACKS
print("\n" + "=" * 60)
print("TEST 3: COMPOSITIONAL ATTACKS (Novel Threat Model)")
print("=" * 60)

scenarios = [
    (
        "Low-Light PGD",
        lambda s: Compose([
            BrightnessStrategy(brightness_factor=0.3 + 0.7 * (1 - s)),
            PGDStrategy(eps=s * 0.01, eps_step=max(s * 0.001, 1e-6), max_iter=10)
        ])
    ),
    (
        "Blurred PGD",
        lambda s: Compose([
            GaussianBlurStrategy(sigma=s * 2.0),
            PGDStrategy(eps=s * 0.01, eps_step=max(s * 0.001, 1e-6), max_iter=10)
        ])
    ),
    (
        "Noisy PGD",
        lambda s: Compose([
            GaussianNoiseStrategy(std_dev=s * 0.03),
            PGDStrategy(eps=s * 0.01, eps_step=max(s * 0.001, 1e-6), max_iter=10)
        ])
    ),
]

for scenario_name, composition in scenarios:
    print(f"\n{scenario_name}:")
    
    for model_name, model in [("vanilla", vanilla), ("robust", robust)]:
        report = search(
            model=model,
            data=samples,
            strategy=composition,
            level_lo=0.0,
            level_hi=1.0,
            device=device,
            normalization="imagenet",
            strategy_name=scenario_name
        )
        results[model_name][scenario_name] = report
        print(f"  {model_name}: threshold={report.search.get('threshold', 'N/A'):.4f}")

In [ ]:
# Cell 9: Compute normalized robustness and create comprehensive report

# Max thresholds for normalization
max_thresholds = {
    "natural": 10.0,
    "adversarial": 0.03,
    "Low-Light PGD": 1.0,
    "Blurred PGD": 1.0,
    "Noisy PGD": 1.0,
}

def get_threshold(report):
    if hasattr(report, 'search') and report.search:
        return report.search.get('threshold', 0)
    return 0

def normalized_robustness(threshold, max_thresh):
    return min(100.0, (threshold / max_thresh) * 100) if max_thresh > 0 else 0

# Build comprehensive results
comprehensive_results = {
    "experiment": {
        "name": "Adversarial Training Effectiveness",
        "dataset": "ImageNet",
        "num_samples": len(samples),
        "device": device,
        "models": {
            "vanilla": "ResNet50 (IMAGENET1K_V2)",
            "robust": "Salman2020Do_R50 (RobustBench, Linf)"
        }
    },
    "tests": {},
    "timestamp": datetime.now().isoformat()
}

print("\n" + "=" * 85)
print("ADVERSARIAL TRAINING EFFECTIVENESS - FINAL RESULTS")
print("=" * 85)
print(f"Dataset: ImageNet | Samples: {len(samples)} | Device: {device}")
print("\nMetric: Normalized Robustness (threshold / max_threshold × 100%)")
print("Higher = more robust (tolerates larger perturbations)\n")

print("Threat Model          | Vanilla |  Robust | Improvement | Thresholds")
print("-" * 85)

for test_name in ["natural", "adversarial", "Low-Light PGD", "Blurred PGD", "Noisy PGD"]:
    v_report = results["vanilla"][test_name]
    r_report = results["robust"][test_name]
    
    v_thresh = get_threshold(v_report)
    r_thresh = get_threshold(r_report)
    max_thresh = max_thresholds[test_name]
    
    v_robust = normalized_robustness(v_thresh, max_thresh)
    r_robust = normalized_robustness(r_thresh, max_thresh)
    improvement = r_robust - v_robust
    
    display_name = "Natural (Blur)" if test_name == "natural" else \
                  "Adversarial (PGD)" if test_name == "adversarial" else test_name
    
    # Store in comprehensive results
    comprehensive_results["tests"][test_name] = {
        "display_name": display_name,
        "max_threshold": max_thresh,
        "vanilla": {
            "threshold": v_thresh,
            "normalized_robustness": round(v_robust, 2),
            "pass_rate": round(v_report.score, 2),
            "runtime": round(v_report.runtime, 2)
        },
        "robust": {
            "threshold": r_thresh,
            "normalized_robustness": round(r_robust, 2),
            "pass_rate": round(r_report.score, 2),
            "runtime": round(r_report.runtime, 2)
        },
        "improvement": round(improvement, 2),
        "improvement_ratio": round(r_thresh / v_thresh, 2) if v_thresh > 0 else None
    }
    
    emoji = "✓" if improvement >= 10 else "✗"
    thresh_str = f"v:{v_thresh:.4f} r:{r_thresh:.4f}"
    print(f"{display_name:22} | {v_robust:6.1f}% | {r_robust:6.1f}% |   {improvement:+6.1f}% {emoji} | {thresh_str}")

# Summary
adv_improvement = comprehensive_results["tests"]["adversarial"]["improvement"]
comp_improvements = [comprehensive_results["tests"][t]["improvement"]
                    for t in ["Low-Light PGD", "Blurred PGD", "Noisy PGD"]]
avg_comp_improvement = sum(comp_improvements) / len(comp_improvements)

print("\n" + "=" * 85)
print("KEY FINDINGS:")
print("-" * 85)
print(f"  • Adversarial (PGD) improvement:     {adv_improvement:+.1f}%")
print(f"  • Compositional average improvement: {avg_comp_improvement:+.1f}%")
print(f"  • Gap: {adv_improvement - avg_comp_improvement:.1f}% less protection on compositional")
print()
print("  CONCLUSION: Adversarial training provides strong protection against")
print("  pure PGD attacks, but LIMITED protection against compositional attacks.")
print("  Standard adversarial training is INSUFFICIENT for real-world robustness!")
print("=" * 85)

comprehensive_results["summary"] = {
    "adversarial_improvement": round(adv_improvement, 2),
    "compositional_avg_improvement": round(avg_comp_improvement, 2),
    "protection_gap": round(adv_improvement - avg_comp_improvement, 2),
    "conclusion": "Adversarial training provides strong protection against pure PGD but limited protection against compositional attacks"
}

In [ ]:
# Cell 10: Save and download results
from google.colab import files

# Save comprehensive JSON
with open('experiment_results_comprehensive.json', 'w') as f:
    json.dump(comprehensive_results, f, indent=2)

print("Saved: experiment_results_comprehensive.json")
print("\nDownloading...")
files.download('experiment_results_comprehensive.json')

In [ ]:
# Cell 11: Generate visualization
import matplotlib.pyplot as plt
import numpy as np

# Extract data
categories = ['Natural\n(Blur)', 'Adversarial\n(PGD)', 'Low-Light\nPGD', 'Blurred\nPGD', 'Noisy\nPGD']
test_keys = ['natural', 'adversarial', 'Low-Light PGD', 'Blurred PGD', 'Noisy PGD']

vanilla_scores = [comprehensive_results['tests'][k]['vanilla']['normalized_robustness'] for k in test_keys]
robust_scores = [comprehensive_results['tests'][k]['robust']['normalized_robustness'] for k in test_keys]

x = np.arange(len(categories))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 7))
bars1 = ax.bar(x - width/2, vanilla_scores, width, label='Vanilla ResNet50', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, robust_scores, width, label='Adversarially-Trained (Salman2020)', color='#27ae60', alpha=0.8)

ax.set_ylabel('Normalized Robustness (%)', fontsize=12)
ax.set_xlabel('Threat Model', fontsize=12)
ax.set_title('Adversarial Training Effectiveness on ImageNet\n(Higher = More Robust, Based on Failure Threshold)', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(categories)
ax.legend(loc='upper left')
ax.set_ylim(0, 110)

# Add value labels
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.1f}%', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=10)

# Add dividing line
ax.axvline(x=1.5, color='gray', linestyle='--', alpha=0.5)
ax.text(0.5, 105, 'Standard Threats', ha='center', fontsize=11, color='gray')
ax.text(3, 105, 'Compositional Threats (Novel)', ha='center', fontsize=11, color='gray')

# Add improvement annotations
for i, (v, r) in enumerate(zip(vanilla_scores, robust_scores)):
    improvement = r - v
    color = 'green' if improvement > 10 else 'red'
    ax.annotate(f'{improvement:+.0f}%', xy=(i, max(v, r) + 8),
                ha='center', fontsize=9, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig('adversarial_training_effectiveness.png', dpi=150, bbox_inches='tight')
plt.show()

files.download('adversarial_training_effectiveness.png')
print("Figure downloaded!")

In [ ]:
# Cell 12: Print JSON for easy copy
print("\n" + "=" * 60)
print("COMPREHENSIVE RESULTS JSON:")
print("=" * 60)
print(json.dumps(comprehensive_results, indent=2))